In [1]:
import pandas as pd
from utils import RESULT_FOLDER, compute_summary_stats, load_results_df

ANALYSIS_MODEL = "haiku-4-5"

model_df = load_results_df(RESULT_FOLDER / ANALYSIS_MODEL)
model_stats = compute_summary_stats(model_df)

print(f"Analysis: {ANALYSIS_MODEL}")
print(f"{'=' * 60}")
print(f"Runs: {model_stats['n_runs']} | Instances: {model_stats['n_instances']}")
print(f"Average execution time: {model_stats['avg_execution_time']:.1f}s ({model_stats['avg_execution_time'] / 60:.1f}m)")
print(f"Mean resolved: {model_stats['mean_resolved']:.2f}%")

# Prepare DataFrame for detailed analysis (drop common metadata columns)
model_df = model_df.drop(columns=["category", "timeout", "error_message", "agent_name", "model", "experiment", "generated_patch"])
model_df.head(n=1)

Analysis: haiku-4-5
Runs: 11 | Instances: 55
Average execution time: 149.7s (2.5m)
Mean resolved: 45.12%


,run_id,instance_id,project,resolved,build,execution_time,llm_duration,turn_count,prompt_tokens,completion_tokens,report_intent,view,glob,grep,edit,powershell,create,read_powershell,stop_powershell,bash
0,20104171950,microsoft__BCApps-4699,Shopify,True,True,110.779,103.416,37,1000000,9300,5,17,4.0,11.0,3.0,1.0,NaN,NaN,NaN,NaN


In [2]:
from bcbench.config import get_config
from bcbench.dataset import DatasetEntry, load_dataset_entries

_config = get_config()
bcbench_dataset: list[DatasetEntry] = load_dataset_entries(_config.paths.dataset_path)

dataset_df = pd.DataFrame(
    [
        {
            "instance_id": entry.instance_id,
            "image_count": entry.metadata.image_count or 0,
            "area": entry.metadata.area or "Unknown",
            "gold_patch": entry.patch,
        }
        for entry in bcbench_dataset
    ]
)

merged_df = model_df.merge(dataset_df, on="instance_id", how="left")
print(f"Merged {len(merged_df)} results with dataset metadata")
merged_df.head(n=1)

Merged 605 results with dataset metadata


,run_id,instance_id,project,resolved,build,execution_time,llm_duration,turn_count,prompt_tokens,completion_tokens,...,grep,edit,powershell,create,read_powershell,stop_powershell,bash,image_count,area,gold_patch
0,20104171950,microsoft__BCApps-4699,Shopify,True,True,110.779,103.416,37,1000000,9300,...,11.0,3.0,1.0,NaN,NaN,NaN,NaN,5,shopify,diff --git a/src/Apps/W1/Shopify/App/src/Produ...


In [3]:
from unidiff import PatchSet


def count_files_in_patch(patch: str) -> int:
    return len(PatchSet(patch))


merged_df["expected_files"] = merged_df["gold_patch"].apply(count_files_in_patch)

bins = [0, 1, 2, float("inf")]
labels = ["1", "2", "3+"]
merged_df["files_bin"] = pd.cut(merged_df["expected_files"], bins=bins, labels=labels)

instance_files_df = (
    merged_df.groupby("instance_id")
    .agg(
        files_bin=("files_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

files_resolution_df = (
    instance_files_df.groupby("files_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

total_instances = files_resolution_df["count"].sum()
files_resolution_df["% Resolved"] = (files_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
files_resolution_df["Instances"] = files_resolution_df["count"].astype(str) + " (" + (files_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
files_resolution_df = files_resolution_df.rename(columns={"files_bin": "Files Changed"})

print('Average "% Resolved" by Number of Files Changed (gold patch):')
print(files_resolution_df[["Files Changed", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Number of Files Changed (gold patch):
Files Changed % Resolved  Instances
            1      49.9% 47 (85.5%)
            2       7.3%   5 (9.1%)
           3+      33.3%   3 (5.5%)


In [4]:
def count_loc_in_patch(patch: str) -> int:
    patch_set = PatchSet(patch)
    return patch_set.added + patch_set.removed


merged_df["expected_loc"] = merged_df["gold_patch"].apply(count_loc_in_patch)

bins = [0, 5, 10, 50, float("inf")]
labels = ["1-5", "6-10", "11-50", "50+"]
merged_df["loc_bin"] = pd.cut(merged_df["expected_loc"], bins=bins, labels=labels)

instance_loc_df = (
    merged_df.groupby("instance_id")
    .agg(
        loc_bin=("loc_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

loc_resolution_df = (
    instance_loc_df.groupby("loc_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

loc_resolution_df["% Resolved"] = (loc_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
loc_resolution_df["Instances"] = loc_resolution_df["count"].astype(str) + " (" + (loc_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
loc_resolution_df = loc_resolution_df.rename(columns={"loc_bin": "LoC Changed"})

print('Average "% Resolved" by Lines of Code Changed (gold patch):')
print(loc_resolution_df[["LoC Changed", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Lines of Code Changed (gold patch):
LoC Changed % Resolved  Instances
        1-5      64.5% 22 (40.0%)
       6-10      42.4%  6 (10.9%)
      11-50      33.8% 21 (38.2%)
        50+      16.7%  6 (10.9%)


In [5]:
merged_df["project_group"] = merged_df["project"].apply(lambda x: "BaseApp" if x == "BaseApp" else "First-party Apps")

instance_project_df = (
    merged_df.groupby("instance_id")
    .agg(
        project_group=("project_group", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

project_resolution_df = (
    instance_project_df.groupby("project_group", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

project_resolution_df["% Resolved"] = (project_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
project_resolution_df["Instances"] = project_resolution_df["count"].astype(str) + " (" + (project_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
project_resolution_df = project_resolution_df.rename(columns={"project_group": "Project Group"})

print('Average "% Resolved" by Project Group:')
print(project_resolution_df[["Project Group", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Project Group:
   Project Group % Resolved  Instances
         BaseApp      44.2% 45 (81.8%)
First-party Apps      49.1% 10 (18.2%)


In [6]:
bins = [-1, 0, 5, 10, float("inf")]
labels = ["0", "1-5", "6-10", "10+"]
merged_df["image_bin"] = pd.cut(merged_df["image_count"], bins=bins, labels=labels)

instance_image_df = (
    merged_df.groupby("instance_id")
    .agg(
        image_bin=("image_bin", "first"),
        resolved_rate=("resolved", "mean"),
    )
    .reset_index()
)

image_resolution_df = (
    instance_image_df.groupby("image_bin", observed=True)
    .agg(
        resolved_rate=("resolved_rate", "mean"),
        count=("instance_id", "count"),
    )
    .reset_index()
)

image_resolution_df["% Resolved"] = (image_resolution_df["resolved_rate"] * 100).round(1).astype(str) + "%"
image_resolution_df["Instances"] = image_resolution_df["count"].astype(str) + " (" + (image_resolution_df["count"] / total_instances * 100).round(1).astype(str) + "%)"
image_resolution_df = image_resolution_df.rename(columns={"image_bin": "Image Count"})

print('Average "% Resolved" by Image Count:')
print(image_resolution_df[["Image Count", "% Resolved", "Instances"]].to_string(index=False))

Average "% Resolved" by Image Count:
Image Count % Resolved  Instances
          0      48.3% 26 (47.3%)
        1-5      42.4%  9 (16.4%)
       6-10      39.9% 13 (23.6%)
        10+      46.8%  7 (12.7%)
